# Phase 1 — Iterators & Generators

**Why this matters for DS/ML:**
- NumPy, Pandas, PyTorch `DataLoader` all use the iterator protocol internally.
- Generators allow you to stream large datasets without loading them fully into memory.
- `itertools` is used constantly in feature engineering and data pipeline construction.

---

## 1. The Iterator Protocol

An **iterable** is any object that can return an iterator (`__iter__`).
An **iterator** is an object that yields values one at a time (`__next__`).

Every `for` loop in Python calls `__iter__` to get an iterator, then calls `__next__` repeatedly.

In [ ]:
# What Python does behind every for loop
my_list = [10, 20, 30]

it = iter(my_list)  # calls my_list.__iter__()
print(next(it))  # calls it.__next__() → 10
print(next(it))  # → 20
print(next(it))  # → 30
# next(it) would now raise StopIteration

In [ ]:
# Build your own iterator class — a counter
class Counter:
    """Yields integers from start up to (but not including) stop."""

    def __init__(self, start, stop):
        self.current = start
        self.stop = stop

    def __iter__(self):
        return self  # the object is its own iterator

    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration
        value = self.current
        self.current += 1
        return value


for n in Counter(1, 5):
    print(n, end=" ")  # 1 2 3 4

In [ ]:
# DS context: iterating over rows of a dataset manually
import numpy as np


class DatasetIterator:
    """Iterates over a 2D numpy array row by row."""

    def __init__(self, data):
        self.data = data
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.data):
            raise StopIteration
        row = self.data[self.index]
        self.index += 1
        return row


data = np.array([[1, 2], [3, 4], [5, 6]])
for row in DatasetIterator(data):
    print(row)

---
## 2. Generator Functions

A generator function uses `yield` instead of `return`. Each call to `next()` resumes execution until the next `yield`.

**Key benefit:** values are produced lazily — only when requested. This is critical for large datasets.

In [ ]:
# Simple generator
def count_up(start, stop):
    while start < stop:
        yield start  # suspends here, returns start
        start += 1


gen = count_up(1, 5)
print(type(gen))  # <class 'generator'>
print(next(gen))  # 1
print(next(gen))  # 2

# Can also consume with a for loop
for n in count_up(10, 13):
    print(n, end=" ")

In [ ]:
# Memory comparison: list vs generator
import sys

n = 1_000_000
list_version = [x**2 for x in range(n)]  # allocates memory for all 1M items
gen_version = (x**2 for x in range(n))  # generator expression, lazy

print(f"List size : {sys.getsizeof(list_version):,} bytes")
print(f"Generator : {sys.getsizeof(gen_version):,} bytes")  # tiny regardless of n

In [ ]:
# DS context: batch generator — core concept behind ML DataLoaders
def batch_generator(data, batch_size=32):
    """Yield successive batches of `batch_size` rows from a numpy array."""
    n_samples = len(data)
    for start in range(0, n_samples, batch_size):
        yield data[start : start + batch_size]


dataset = np.arange(100).reshape(100, 1)  # 100 samples, 1 feature

for i, batch in enumerate(batch_generator(dataset, batch_size=30)):
    print(f"Batch {i + 1}: {len(batch)} samples")

In [ ]:
# Generator expression vs list comprehension
import statistics

scores = [72, 85, 90, 60, 78, 95, 55, 88]

# List comprehension — creates the whole list in memory
squared_list = [x**2 for x in scores if x >= 70]

# Generator expression — computes on demand
squared_gen = (x**2 for x in scores if x >= 70)

# Pass generator directly to sum, statistics functions — no list needed
total = sum(x**2 for x in scores if x >= 70)
mean_val = statistics.mean(x**2 for x in scores if x >= 70)

print(f"Total squared (>=70): {total}")
print(f"Mean squared  (>=70): {mean_val:.2f}")

---
## 3. `itertools` — The Data Pipeline Toolbox

`itertools` gives you composable, memory-efficient tools for working with iterables. Used heavily in feature engineering, cross-validation, and data pipelines.

In [ ]:
import itertools

# --- chain: flatten multiple iterables into one ---
train_ids = [1, 2, 3]
val_ids = [4, 5]
all_ids = list(itertools.chain(train_ids, val_ids))
print("chain:", all_ids)

In [ ]:
# --- product: cartesian product — useful for hyperparameter grid search ---
learning_rates = [0.001, 0.01, 0.1]
batch_sizes = [16, 32, 64]

grid = list(itertools.product(learning_rates, batch_sizes))
print(f"Total combinations: {len(grid)}")
for lr, bs in grid[:4]:
    print(f"  lr={lr}, batch_size={bs}")

In [ ]:
# --- combinations & permutations ---
features = ["age", "income", "score"]

# All 2-feature combinations for pairwise analysis
pairs = list(itertools.combinations(features, 2))
print("Feature pairs:", pairs)

In [ ]:
# --- islice: take first N items from any iterator efficiently ---
def infinite_counter(start=0):
    while True:
        yield start
        start += 1


# Take first 5 without storing anything else
first_five = list(itertools.islice(infinite_counter(100), 5))
print("First 5:", first_five)

In [ ]:
# --- groupby: group consecutive items by a key ---
# Note: data must be sorted by the key first
data = [
    {"category": "A", "value": 10},
    {"category": "A", "value": 20},
    {"category": "B", "value": 5},
    {"category": "B", "value": 15},
    {"category": "C", "value": 30},
]

for category, group in itertools.groupby(data, key=lambda x: x["category"]):
    values = [item["value"] for item in group]
    print(f"Category {category}: values={values}, mean={sum(values) / len(values):.1f}")

---
## 4. `functools` — Functional Tools for DS/ML

`functools` is used in ML for caching expensive computations and creating specialized function variants.

In [ ]:
import functools
import time


# --- lru_cache: memoize expensive function calls ---
@functools.lru_cache(maxsize=128)
def expensive_feature(user_id):
    """Simulate an expensive DB lookup or computation."""
    time.sleep(0.1)  # pretend this is slow
    return user_id * 42


start = time.time()
for uid in [1, 2, 1, 3, 2, 1]:  # 1, 2, 3 called multiple times
    result = expensive_feature(uid)
elapsed = time.time() - start

print(f"Elapsed: {elapsed:.2f}s  (only 3 actual calls, not 6)")
print(expensive_feature.cache_info())

In [ ]:
# --- partial: freeze some arguments of a function ---
def scale_value(value, mean, std):
    """Z-score normalization."""
    return (value - mean) / std


# Create a specialized scaler for a specific feature (mean=50, std=10)
scale_age = functools.partial(scale_value, mean=50, std=10)

ages = [30, 45, 55, 70]
scaled = list(map(scale_age, ages))
print("Z-scores:", [f"{z:.2f}" for z in scaled])

In [ ]:
# --- reduce: aggregate a sequence with a binary function ---
from functools import reduce

numbers = [1, 2, 3, 4, 5]

# Product of all elements
product = reduce(lambda acc, x: acc * x, numbers)
print("Product:", product)  # 120

# Build a pipeline of transformations
transforms = [
    lambda x: x * 2,  # double
    lambda x: x + 10,  # shift
    lambda x: x / 100,  # normalize
]

value = 5
result = reduce(lambda v, fn: fn(v), transforms, value)
print(f"Pipeline result: {result}")  # ((5*2)+10)/100 = 0.2

---
## Summary

| Concept | Key Method/Syntax | DS/ML Use Case |
|---------|-------------------|----------------|
| Iterator | `__iter__`, `__next__` | Custom data loaders |
| Generator function | `yield` | Streaming large datasets, batch iteration |
| Generator expression | `(expr for x in iterable)` | Memory-efficient transformations |
| `itertools.chain` | chains iterables | Merging train/val sets |
| `itertools.product` | cartesian product | Hyperparameter grid search |
| `itertools.combinations` | unique pairs | Pairwise feature analysis |
| `functools.lru_cache` | memoization decorator | Cache expensive feature lookups |
| `functools.partial` | freeze function args | Create specialized transformers |
| `functools.reduce` | cumulative fold | Building function pipelines |